# SME Loan Approval Scoring

### Retail Loan Mortgage Approval — Banking-AI

This notebook explains each step in plain language so new learners can follow:

## Step 0 — Notebook Header

**Prerequisites:** Basic Python (`pandas`, `numpy`, `matplotlib`), familiarity with `scikit-learn` basics, and basic understanding of retail lending and mortgage approval terminology.

**What You Will Learn:**

After completing this notebook, you will be able to:
- Explain the retail lending and mortgage approval problem and why the chosen classification approach suits this banking workflow.
- Implement an end-to-end classification pipeline on synthetic data and evaluate performance with domain-appropriate metrics.
- Assess whether the model is operationally viable, considering error costs, fairness, and the limitations of synthetic data.

**Estimated time:** 35–45 minutes

**Collection context:** Mortgage risk, automated underwriting, appraisal valuation, and loan approval processes in retail banking.

## Step 1 — Banking Problem Introduction

**Why this notebook matters:** SMEs are the backbone of the economy, but they often struggle to get loans because they don't have a simple 'FICO' score. Banks use AI to analyze business bank statements and tax returns to determine if a business can afford to pay back a loan.

**Input data used:** Annual Revenue, Net Profit Margin, Current Ratio (Liquidity), Years in Business, and Debt Service Coverage Ratio (DSCR).

**What we predict:** Credit Score / Approval Likelihood.

**ML method used:** XGBoost (or similar Gradient Boosting) - the gold standard for tabular data.

**Learning goal:** Understand how 'Financial Ratios' are used as features in business lending.

**Primary stakeholders:** mortgage officers, underwriters, credit risk managers, and compliance teams.

**Comprehension check:** Before looking at the data, write one sentence describing the core decision this model supports and who benefits from getting it right.

**Domain framing note:** This notebook uses synthetic data for educational purposes. It demonstrates decision-support prototyping, not production-ready deployment.

## Step 2 — Environment Setup

Import libraries and set deterministic seeds for reproducibility.

In [ ]:
# Requires: numpy>=1.24, pandas>=2.0, scikit-learn>=1.3, matplotlib>=3.7, seaborn>=0.13

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, precision_recall_curve, accuracy_score

sns.set_theme(style="whitegrid")

RANDOM_SEED = 42
RNG = np.random.default_rng(RANDOM_SEED)
plt.rcParams["figure.figsize"] = (10, 6)

print("Environment ready.")


## Step 3 — Data Creation & Context

Build Synthetic SME Data

We create 3,000 businesses with various financial health profiles.

In [ ]:
n = 3000
rev = RNG.uniform(100000, 5000000, n) # Revenue $100k to $5M
profit_margin = RNG.uniform(-0.1, 0.3, n) # -10% to +30%
years_in_biz = RNG.integers(1, 25, n)
dscr = RNG.uniform(0.5, 3.0, n) # Debt Service Coverage Ratio (1.2 is usually the minimum)

# Logic for Creditworthiness:
# Higher margin, more years, and DSCR > 1.2 lead to approval.
score = (
    2.0 * profit_margin + 
    0.5 * np.log10(rev) + 
    0.1 * years_in_biz + 
    1.5 * (dscr - 1.2) + 
    RNG.normal(0, 0.5, n)
)
approved = (score > 1.5).astype(int)

df = pd.DataFrame({
    'revenue': rev,
    'profit_margin': profit_margin,
    'years_in_biz': years_in_biz,
    'dscr': dscr,
    'approved': approved
})

print(f"Dataset created. Approval rate: {df['approved'].mean():.1%}")

## Step 4 — Exploratory Data Analysis

For this workflow, primary exploration happens through the operational visualisations in later steps.

## Step 5 — Feature Engineering

Prepare features for modelling. Domain-meaningful transformations help the model learn patterns aligned with banking practice.

In [ ]:
# Features are used as created in Step 3.
print("Target column: 'approved'")

## Step 6 — Baseline Establishment

A baseline sets the floor that any useful model must beat. Without it, performance numbers have no context.

In [ ]:
# --- Baseline: always predict the majority class ---
baseline_clf = DummyClassifier(strategy='most_frequent', random_state=RANDOM_SEED)
baseline_clf.fit(X_train, y_train)
baseline_pred = baseline_clf.predict(X_test)
baseline_acc = accuracy_score(y_test, baseline_pred)
print(f"Baseline accuracy (majority-class): {baseline_acc:.3f}")
print(f"Any useful model must beat this {baseline_acc:.3f} floor.")

## Step 7 — Model Training & Evaluation

Train the model and compare performance against the baseline.

**Prediction prompt:** Before viewing the results, what performance do you expect and why?

Gradient Boosting handles the complex interactions between revenue and profit.

In [ ]:
X = df.drop('approved', axis=1)
y = df['approved']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_SEED)

model = HistGradientBoostingClassifier()
model.fit(X_train, y_train)

y_proba = model.predict_proba(X_test)[:, 1]

In [ ]:
auc = roc_auc_score(y_test, y_proba)
print(f"Model AUC Score: {auc:.3f}")

precision, recall, _ = precision_recall_curve(y_test, y_proba)
plt.plot(recall, precision)
plt.xlabel('Recall (Approving good businesses)')
plt.ylabel('Precision (Not approving bad businesses)')
plt.title('SME Lending Precision-Recall Curve')
plt.tight_layout()
plt.show()
plt.close()

**Alt-text:** Line chart, precision-recall curve titled "SME Lending Precision-Recall Curve" with Recall (Approving good businesses) on the x-axis and Precision (Not approving bad businesses) on the y-axis. The curve shows the trade-off between detection rate and false-alarm rate at different thresholds.

In [ ]:
from sklearn.inspection import permutation_importance

r = permutation_importance(model, X_test, y_test, n_repeats=10, random_state=RANDOM_SEED)
for i in r.importances_mean.argsort()[::-1]:
    print(f"{X.columns[i]}: {r.importances_mean[i]:.3f}")

## Step 8 — Interpretability & Explainability

Interpret the model in domain terms. A banking professional should be able to assess whether the model's reasoning is credible.

1. **DSCR is King:** For business lending, the Debt Service Coverage Ratio (DSCR) is often the most critical feature. It tells the bank: "Does this business have enough cash left over after expenses to pay their loan?"
2. **Stability:** Years in business acts as a proxy for stability. A business that survived for 10 years is statistically less likely to fail next month than a 6-month-old startup.
3. **Financial Health:** Even a high-revenue business might be rejected if its profit margin is negative, as it's 'burning' cash rather than generating it.

## Step 9 — Operational Application

Demonstrate how the model output feeds into a real banking workflow decision.

In [ ]:
# --- Scenario: score representative cases from the test set ---
print("Operational demonstration — model decision support:\n")
proba = model.predict_proba(X_test)[:, 1]
low_idx  = int(np.argmin(proba))
high_idx = int(np.argmax(proba))
mid_idx  = int(np.argsort(proba)[len(proba) // 2])

for label, idx in [("Low-risk", low_idx), ("Medium-risk", mid_idx), ("High-risk", high_idx)]:
    row = X_test.iloc[idx]
    prob = proba[idx]
    print(f"{label} case  predicted probability: {prob:.1%}")
    print(f"  Features: {dict(row.round(2))}")
    print()

print("A decision-maker would combine these scores with business rules and domain judgement.")

## Step 10 — Summary & Reflection

**What you accomplished:** You built an end-to-end retail lending and mortgage approval workflow — from problem framing through data creation, baseline comparison, model training, evaluation, and operational demonstration.

**Limitations:**

- The dataset is synthetic and cannot capture the full complexity of real banking data.
- Model performance on synthetic data does not guarantee equivalent performance in production.
- This notebook is an educational prototype. Real deployment requires validated data, regulatory review, and ongoing monitoring.

**Ethical reflection:**

- Mortgage models must comply with fair lending laws and avoid discriminatory approval patterns.
- Automated underwriting can disadvantage applicants from historically underserved communities.
- Transparency in loan decisions is a regulatory requirement, not an optional feature.

**Reflection questions:**

1. What additional data sources or features would you need before trusting this model in a live banking environment?
2. How would the relative cost of false positives versus false negatives change the metric you optimise for?
3. How could you adapt this workflow to a related problem in retail lending and mortgage approval?

## References

- Scikit-learn documentation: [https://scikit-learn.org/stable/](https://scikit-learn.org/stable/)
- Project quality baseline: `QUALITY_STANDARD.md`
- Domain context drawn from public banking and financial services literature.